# 01 Data Prep

End-to-end data pipeline (matches `docs/claude-3.md` Step 6):

1. Optional: install Kaggle CLI locally (`pip install kaggle`) and download raw novels.
2. On Kaggle: clone this repo, attach the **Jinyong Wuxia** dataset as input, then run the code cells.
3. `clean_text.py` — raw `data/raw/*.txt` → cleaned `data/processed/*.txt`.
4. `build_instructions.py` — cleaned text → `data/instructions/jinyong_sft.jsonl`.
5. Inspect samples; upload JSONL as a Kaggle Dataset for `02_train.ipynb`.

## Resolve repository root
Works when the notebook lives in `notebooks/` or the repo root.

In [ ]:
from pathlib import Path
import os


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts" / "clean_text.py").is_file():
            return cand
    raise FileNotFoundError("Could not find scripts/clean_text.py; cd into jinyong-finetune")


REPO = find_repo_root()
os.chdir(REPO)
print("REPO:", REPO)

## Kaggle-only (uncomment)
```
# !pip install -q pyyaml
# !git clone https://github.com/jxjwilliam/jinyong-finetune.git /kaggle/working/jinyong-finetune
# %cd /kaggle/working/jinyong-finetune
```
Mount the Kaggle dataset so `data/raw/` contains the `.txt` novels (or set `--input-dir`).

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/clean_text.py", "--dry-run"], check=False)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/clean_text.py"], check=False)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/build_instructions.py", "--stats"], check=False)

## Inspect JSONL samples

In [ ]:
import json
import random
from pathlib import Path

p = Path("data/instructions/jinyong_sft.jsonl")
if not p.is_file():
    print("Run build_instructions first.")
else:
    lines = p.read_text(encoding="utf-8").splitlines()
    sample = random.sample(lines, k=min(5, len(lines)))
    for row in sample:
        obj = json.loads(row)
        print("instruction:", obj["instruction"][:60], "…")
        print("input len:", len(obj["input"]), "output len:", len(obj["output"]))
        print("---")